In [22]:
import pandas as pd


In [23]:
df_pms = pd.read_csv("../data/processed/player_match_stats.csv")
df_events = pd.read_csv("../data/processed/events.csv")

In [24]:
df_matches = pd.read_csv("../data/processed/matches.csv")

In [25]:
end_events = df_events[df_events["type"] == "End"]
match_duration = end_events.groupby("match_id")["minute"].max().reset_index()
match_duration = match_duration.rename(columns={"minute": "match_duration"})

df_matches = df_matches.merge(match_duration, left_on="game_id", right_on="match_id", how="left")
df_matches = df_matches.drop(columns=["match_id"])

In [26]:
df_matches

,game_id,home_team_id,away_team_id,matchday,home_score,away_score,match_date,match_duration
0,1911273,313,249,1,1,0,2025-08-15,97
1,1911284,309,228,1,0,1,2025-08-16,95
2,1911290,248,217,1,3,1,2025-08-16,98
3,1911296,613,246,1,0,1,2025-08-16,98
4,1911275,614,2832,1,1,0,2025-08-17,96
...,...,...,...,...,...,...,...,...
301,1911524,249,313,34,3,1,2026-05-17,96
302,1911525,302,246,34,0,0,2026-05-17,21
303,1911526,613,314,34,0,0,2026-05-17,94
304,1911527,2832,304,34,2,1,2026-05-17,95


In [27]:
substitution_on = df_events[df_events["type"] == "SubstitutionOn"][["player_id", "match_id", "minute"]].rename(columns={"minute": "on_minute"})
substitution_off = df_events[df_events["type"] == "SubstitutionOff"][["player_id", "match_id", "minute"]].rename(columns={"minute": "off_minute"})
red_cards = df_events[(df_events["type"] == "Card") & (df_events["card_type"].isin(["Red", "SecondYellow"]))][["player_id", "match_id", "minute"]].rename(columns={"minute": "card_minute"})

df_pms = df_pms.merge(substitution_on, on=["player_id", "match_id"], how="left")
df_pms = df_pms.merge(substitution_off, on=["player_id", "match_id"], how="left")
df_pms = df_pms.merge(red_cards, on=["player_id", "match_id"], how="left")

df_pms = df_pms.merge(df_matches[["game_id", "match_duration"]], left_on="match_id", right_on="game_id", how="left")
df_pms = df_pms.drop(columns=["game_id"])

df_pms["start_minute"] = df_pms["on_minute"].fillna(0)
df_pms["end_minute"] = df_pms[["off_minute", "card_minute", "match_duration"]].min(axis=1)
df_pms["minutes_played"] = (df_pms["end_minute"] - df_pms["start_minute"]).clip(lower=0)

df_pms = df_pms.drop(columns=["on_minute", "off_minute", "card_minute", "start_minute", "end_minute"])

In [28]:
print(df_pms.shape) 
print(df_pms["minutes_played"].describe())  # la plupart entre 0 et 100
print(df_pms[df_pms["minutes_played"] > 100])  # doit être vide ou quasi

(9414, 28)
count    9414.000000
mean       67.985022
std        31.114844
min         0.000000
25%        37.000000
50%        80.000000
75%        95.000000
max       102.000000
Name: minutes_played, dtype: float64
      player_id  match_id  team_id  goals  own_goals  assists  key_passes  \
28        10454   1911425      246      0          0        0           1   
37        10454   1911502      246      0          0        0           1   
386       91926   1911387      314      0          0        0           0   
396       91926   1911502      314      0          0        0           1   
649      105151   1911387     2332      0          0        0           1   
...         ...       ...      ...    ...        ...      ...         ...   
8519     530864   1911387      314      0          0        1           1   
8761     541331   1911445      309      0          0        0           0   
8830     547575   1911387      314      0          0        0           0   
8927     55517

In [29]:
dup = df_pms[df_pms.duplicated(subset=["player_id", "match_id"], keep=False)]
print(dup[["player_id", "match_id"]])

      player_id  match_id
6663     445649   1911459
6664     445649   1911459


In [36]:
df_pms = pd.read_csv("../data/processed/player_match_stats.csv")

substitution_on = df_events[df_events["type"] == "SubstitutionOn"][["player_id", "match_id", "minute"]]
substitution_off = df_events[df_events["type"] == "SubstitutionOff"][["player_id", "match_id", "minute"]]
red_cards = df_events[(df_events["type"] == "Card") & (df_events["card_type"].isin(["Red", "SecondYellow"]))][["player_id", "match_id", "minute"]]

substitution_on = substitution_on.sort_values("minute").drop_duplicates(subset=["player_id", "match_id"], keep="first")
substitution_off = substitution_off.sort_values("minute").drop_duplicates(subset=["player_id", "match_id"], keep="first")
red_cards = red_cards.sort_values("minute").drop_duplicates(subset=["player_id", "match_id"], keep="first")

substitution_on = substitution_on.rename(columns={"minute": "on_minute"})
substitution_off = substitution_off.rename(columns={"minute": "off_minute"})
red_cards = red_cards.rename(columns={"minute": "card_minute"})

df_pms = df_pms.merge(substitution_on, on=["player_id", "match_id"], how="left")
df_pms = df_pms.merge(substitution_off, on=["player_id", "match_id"], how="left")
df_pms = df_pms.merge(red_cards, on=["player_id", "match_id"], how="left")

df_pms = df_pms.merge(df_matches[["game_id", "match_duration"]], left_on="match_id", right_on="game_id", how="left")
df_pms = df_pms.drop(columns=["game_id"])

df_pms["start_minute"] = df_pms["on_minute"].fillna(0)
df_pms["end_minute"] = df_pms[["off_minute", "card_minute", "match_duration"]].min(axis=1)
df_pms["minutes_played"] = (df_pms["end_minute"] - df_pms["start_minute"]).clip(lower=0)

df_pms = df_pms.drop(columns=["on_minute", "off_minute", "card_minute", "start_minute", "end_minute"])

print(df_pms.shape)
print(df_pms.duplicated(subset=["player_id", "match_id"]).sum())

Int64
Int64


In [37]:
df_matches["match_duration"] = df_matches["match_duration"].astype("Int64")
df_matches.to_csv("../data/processed/matches.csv", index=False)

df_pms["minutes_played"] = df_pms["minutes_played"].astype("Int64")
df_pms.to_csv("../data/processed/player_match_stats.csv", index=False)

print(df_matches["match_duration"].dtype)
print(df_pms["minutes_played"].dtype)

Int64
Int64


In [39]:
from dotenv import load_dotenv
import os
from supabase import create_client
import pandas as pd

load_dotenv("../.env", override=True)
url = os.getenv("SUPABASE_URL")
service_key = os.getenv("SUPABASE_SERVICE_KEY")
supabase = create_client(url, service_key)

# football_match : mise à jour de match_duration uniquement
df_matches = pd.read_csv("../data/processed/matches.csv")
matches_subset = df_matches[["game_id", "match_duration"]]
matches_records = matches_subset.astype(object).where(matches_subset.notna(), None).to_dict(orient="records")
supabase.table("football_match").upsert(matches_records, on_conflict="game_id").execute()

# player_match_stats : mise à jour de minutes_played uniquement
df_pms = pd.read_csv("../data/processed/player_match_stats.csv")
pms_subset = df_pms[["player_id", "match_id", "minutes_played"]]
pms_records = pms_subset.astype(object).where(pms_subset.notna(), None).to_dict(orient="records")
supabase.table("player_match_stats").upsert(pms_records, on_conflict="player_id,match_id").execute()

print("Mise à jour terminée")

APIError: {'message': 'null value in column "home_team_id" of relation "football_match" violates not-null constraint', 'code': '23502', 'hint': None, 'details': 'Failing row contains (1911273, null, null, null, null, null, null, 97).'}

In [40]:
df_matches[["game_id", "match_duration"]].to_csv("../data/processed/staging_match_duration.csv", index=False)
df_pms[["player_id", "match_id", "minutes_played"]].to_csv("../data/processed/staging_minutes_played.csv", index=False)